<a href="https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karim-yasser/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [43]:
%pip -q install duckdb huggingface_hub


In [44]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [45]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [46]:
print("Token exists:", bool(HF_TOKEN))
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

Token exists: True
Token length: 37


In [47]:
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


In [48]:
from getpass import getpass

HF_TOKEN = getpass("Paste the full Hugging Face token: ")
print("Token length:", len(HF_TOKEN))

Paste the full Hugging Face token: ··········
Token length: 29


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [49]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [50]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224


In [51]:

features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_last30,

            SUM(CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.report_date > b.end_d - INTERVAL 60 DAY
                THEN f.gsc_impressions ELSE 0
            END) AS imp_prev30,

            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks ELSE 0
            END) AS clk_prev30,

            AVG(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END) AS pos_prev30

        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
          AND f.report_date <= b.end_d
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print("90-day window rows:", len(features_90))
print("Unique clients:", features_90["client_hash_id"].nunique())
features_90.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

90-day window rows: 111247
Unique clients: 50


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,172.0,0.0,22.406195
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,301.0,0.0,24.094002
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,147.0,0.0,24.732526
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,239.0,1.0,21.121470
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,745.0,2.0,24.027144


In [53]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# Build the current label and features
data = features.copy()
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = ["imp_prev30", "clk_last30", "pos_last30"]

model_data = data.dropna(subset=feature_cols + ["client_hash_id"])
X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1,
    class_weight="balanced"
)
model.fit(X_train, y_train)

print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print("Overlapping clients:",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print(classification_report(y_test, model.predict(X_test), digits=3))


Train clients: 36
Test clients: 12
Overlapping clients: 0


KeyboardInterrupt: 

## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


In [ ]:
print("Query table columns:")
print(con.sql(f"""
    SELECT *
    FROM {TABLES['fact_query_90d']}
    LIMIT 0
""").df().columns.tolist())

print("\nQuery table sample:")
print(con.sql(f"""
    SELECT *
    FROM {TABLES['fact_query_90d']}
    LIMIT 5
""").df().to_string(index=False))

In [ ]:
print(con.sql(f"""
    SELECT
        window_start,
        window_end,
        COUNT(*) AS query_rows,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {TABLES['fact_query_90d']}
    GROUP BY window_start, window_end
    ORDER BY window_end, window_start
""").df().to_string(index=False))

In [ ]:
print(con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM {TABLES['fact_daily']}
    GROUP BY 1
    ORDER BY 1
""").df().to_string(index=False))


In [ ]:
print(con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(DISTINCT report_date) AS observed_days,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    GROUP BY 1
    ORDER BY 1
""").df().to_string(index=False))

In [ ]:
june_data = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-04-01'
          AND report_date <= DATE '2026-06-30'
        GROUP BY 1, 2, 3
    ),
    monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date >= DATE '2026-04-01'
                      AND report_date < DATE '2026-05-01'
                     THEN impressions ELSE 0 END) AS imp_apr,
            SUM(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN impressions ELSE 0 END) AS imp_may,
            SUM(CASE WHEN report_date >= DATE '2026-06-01'
                      AND report_date < DATE '2026-07-01'
                     THEN impressions ELSE 0 END) AS imp_jun,
            SUM(CASE WHEN report_date >= DATE '2026-04-01'
                      AND report_date < DATE '2026-05-01'
                     THEN clicks ELSE 0 END) AS clicks_apr,
            SUM(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN clicks ELSE 0 END) AS clicks_may,
            AVG(CASE WHEN report_date >= DATE '2026-04-01'
                      AND report_date < DATE '2026-05-01'
                     THEN avg_position END) AS pos_apr,
            AVG(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN avg_position END) AS pos_may,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-04-01'
                                 AND report_date < DATE '2026-05-01'
                                THEN report_date END) AS days_apr,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-05-01'
                                 AND report_date < DATE '2026-06-01'
                                THEN report_date END) AS days_may,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-06-01'
                                 AND report_date < DATE '2026-07-01'
                                THEN report_date END) AS days_jun
        FROM daily
        GROUP BY 1, 2
    )
    SELECT *,
        CASE WHEN imp_jun < 0.8 * imp_may THEN 1 ELSE 0 END AS is_declining
    FROM monthly
    WHERE days_apr >= 28
      AND days_may >= 28
      AND days_jun >= 28
      AND imp_may >= 100
""").df()

print("June evaluation rows:", len(june_data))
print("Unique clients:", june_data["client_hash_id"].nunique())
print("\nLabel distribution:")
print(june_data["is_declining"].value_counts(normalize=True).rename("share"))
june_data.head()

In [ ]:
train_data = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2025-02-01'
          AND report_date < DATE '2026-06-01'
        GROUP BY 1, 2, 3
    ),
    monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            DATE_TRUNC('month', report_date) AS month,
            SUM(impressions) AS impressions,
            SUM(clicks) AS clicks,
            AVG(avg_position) AS avg_position,
            COUNT(DISTINCT report_date) AS observed_days
        FROM daily
        GROUP BY 1, 2, 3
    ),
    lagged AS (
        SELECT
            *,
            LAG(impressions, 1) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev_impressions,
            LAG(impressions, 2) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev2_impressions,
            LAG(clicks, 1) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev_clicks,
            LAG(avg_position, 1) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev_position,
            LAG(observed_days, 1) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev_days,
            LAG(observed_days, 2) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY month
            ) AS prev2_days
        FROM monthly
    )
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        prev_impressions,
        prev2_impressions,
        prev_clicks,
        prev_position,
        impressions AS target_impressions,
        CASE WHEN impressions < 0.8 * prev_impressions
             THEN 1 ELSE 0 END AS is_declining
    FROM lagged
    WHERE month >= DATE '2025-04-01'
      AND month < DATE '2026-06-01'
      AND observed_days >= 28
      AND prev_days >= 28
      AND prev2_days >= 28
      AND prev_impressions >= 100
""").df()

print("Training rows:", len(train_data))
print("Training clients:", train_data["client_hash_id"].nunique())
print("\nTraining label distribution:")
print(train_data["is_declining"].value_counts(normalize=True).rename("share"))
print("\nMonth range:")
print(train_data["month"].min(), "to", train_data["month"].max())

In [ ]:

june_data = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date >= DATE '2026-04-01'
          AND report_date < DATE '2026-07-01'
        GROUP BY 1, 2, 3
    ),
    monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date >= DATE '2026-04-01'
                      AND report_date < DATE '2026-05-01'
                     THEN impressions ELSE 0 END) AS imp_apr,
            SUM(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN impressions ELSE 0 END) AS imp_may,
            SUM(CASE WHEN report_date >= DATE '2026-06-01'
                      AND report_date < DATE '2026-07-01'
                     THEN impressions ELSE 0 END) AS imp_jun,
            SUM(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN clicks ELSE 0 END) AS clicks_may,
            AVG(CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN avg_position END) AS pos_may,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-04-01'
                      AND report_date < DATE '2026-05-01'
                     THEN report_date END) AS days_apr,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-05-01'
                      AND report_date < DATE '2026-06-01'
                     THEN report_date END) AS days_may,
            COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-06-01'
                      AND report_date < DATE '2026-07-01'
                     THEN report_date END) AS days_jun
        FROM daily
        GROUP BY 1, 2
    )
    SELECT *,
        CASE WHEN imp_jun < 0.8 * imp_may THEN 1 ELSE 0 END
            AS is_declining
    FROM monthly
    WHERE days_apr >= 28
      AND days_may >= 28
      AND days_jun >= 28
      AND imp_may >= 100
""").df()

print("June test rows:", f"{len(june_data):,}")
print("Unique clients:", june_data["client_hash_id"].nunique())
print("\nLabel distribution:")
print(june_data["is_declining"].value_counts(normalize=True).round(3))


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

feature_cols = [
    "prev_impressions",
    "prev2_impressions",
    "prev_clicks",
    "prev_position"
]

X_train = train_data[feature_cols].fillna(0)
y_train = train_data["is_declining"]

X_test = june_data[
    ["imp_may", "imp_apr", "clicks_may", "pos_may"]
].copy()

X_test.columns = feature_cols
X_test = X_test.fillna(0)
y_test = june_data["is_declining"]

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("June 2026 Evaluation")
print("--------------------")
print("Test rows:", f"{len(y_test):,}")
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, predictions, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
import pandas as pd

# Simple baseline: predict every content item as declining
baseline_predictions = [1] * len(y_test)

results = pd.DataFrame([
    {
        "Model": "Baseline (all declining)",
        "Accuracy": accuracy_score(y_test, baseline_predictions),
        "Precision": precision_score(y_test, baseline_predictions, zero_division=0),
        "Recall": recall_score(y_test, baseline_predictions, zero_division=0),
        "F1": f1_score(y_test, baseline_predictions, zero_division=0)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0)
    }
])

print("Baseline vs Model")
display(results.round(4))

print("Random Forest ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))
print("Random Forest Average Precision:",
      round(average_precision_score(y_test, probabilities), 4))

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importance")
display(importance)


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Use earlier months for training and later months for validation
train_mask = train_data["month"] < "2026-03-01"
valid_mask = train_data["month"] >= "2026-03-01"

X_train_cv = train_data.loc[train_mask, feature_cols].fillna(0)
y_train_cv = train_data.loc[train_mask, "is_declining"]

X_valid_cv = train_data.loc[valid_mask, feature_cols].fillna(0)
y_valid_cv = train_data.loc[valid_mask, "is_declining"]

candidates = [
    {"max_depth": 5, "min_samples_leaf": 20},
    {"max_depth": 10, "min_samples_leaf": 20},
    {"max_depth": 15, "min_samples_leaf": 50},
    {"max_depth": None, "min_samples_leaf": 50}
]

results = []

for params in candidates:
    candidate = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        **params
    )

    candidate.fit(X_train_cv, y_train_cv)
    valid_pred = candidate.predict(X_valid_cv)
    valid_prob = candidate.predict_proba(X_valid_cv)[:, 1]

    results.append({
        "max_depth": str(params["max_depth"]),
        "min_samples_leaf": params["min_samples_leaf"],
        "accuracy": accuracy_score(y_valid_cv, valid_pred),
        "precision": precision_score(y_valid_cv, valid_pred, zero_division=0),
        "recall": recall_score(y_valid_cv, valid_pred, zero_division=0),
        "f1": f1_score(y_valid_cv, valid_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_valid_cv, valid_prob)
    })

validation_results = pd.DataFrame(results)
display(validation_results.round(4))


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

baseline_pred = [1] * len(y_valid_cv)

best_candidate = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_candidate.fit(X_train_cv, y_train_cv)

valid_pred = best_candidate.predict(X_valid_cv)
valid_prob = best_candidate.predict_proba(X_valid_cv)[:, 1]

comparison = pd.DataFrame([
    {
        "Model": "Baseline (all declining)",
        "Accuracy": accuracy_score(y_valid_cv, baseline_pred),
        "Precision": precision_score(y_valid_cv, baseline_pred, zero_division=0),
        "Recall": recall_score(y_valid_cv, baseline_pred, zero_division=0),
        "F1": f1_score(y_valid_cv, baseline_pred, zero_division=0)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_valid_cv, valid_pred),
        "Precision": precision_score(y_valid_cv, valid_pred, zero_division=0),
        "Recall": recall_score(y_valid_cv, valid_pred, zero_division=0),
        "F1": f1_score(y_valid_cv, valid_pred, zero_division=0)
    }
])

display(comparison.round(4))
print("Validation ROC-AUC:", round(roc_auc_score(y_valid_cv, valid_prob), 4))


In [ ]:

train_enhanced = train_data.copy()

train_enhanced["impression_change"] = (
    train_enhanced["prev_impressions"] -
    train_enhanced["prev2_impressions"]
) / (train_enhanced["prev2_impressions"] + 1)

train_enhanced["log_prev_impressions"] = __import__("numpy").log1p(
    train_enhanced["prev_impressions"].clip(lower=0)
)

train_enhanced["log_prev2_impressions"] = __import__("numpy").log1p(
    train_enhanced["prev2_impressions"].clip(lower=0)
)

train_enhanced["clicks_per_impression"] = (
    train_enhanced["prev_clicks"] /
    (train_enhanced["prev_impressions"] + 1)
)

print("Enhanced training rows:", f"{len(train_enhanced):,}")
print("\nNew features:")
print([
    "impression_change",
    "log_prev_impressions",
    "log_prev2_impressions",
    "clicks_per_impression"
])


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

new_features = feature_cols + [
    "impression_change",
    "log_prev_impressions",
    "log_prev2_impressions",
    "clicks_per_impression"
]

train_mask = train_enhanced["month"] < "2026-03-01"
valid_mask = train_enhanced["month"] >= "2026-03-01"

X_train_new = train_enhanced.loc[train_mask, new_features].fillna(0)
y_train_new = train_enhanced.loc[train_mask, "is_declining"]

X_valid_new = train_enhanced.loc[valid_mask, new_features].fillna(0)
y_valid_new = train_enhanced.loc[valid_mask, "is_declining"]

enhanced_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

enhanced_model.fit(X_train_new, y_train_new)

valid_pred_new = enhanced_model.predict(X_valid_new)
valid_prob_new = enhanced_model.predict_proba(X_valid_new)[:, 1]

print("Enhanced Model Validation Results")
print("---------------------------------")
print("Accuracy:", round(accuracy_score(y_valid_new, valid_pred_new), 4))
print("Precision:", round(precision_score(y_valid_new, valid_pred_new, zero_division=0), 4))
print("Recall:", round(recall_score(y_valid_new, valid_pred_new, zero_division=0), 4))
print("F1:", round(f1_score(y_valid_new, valid_pred_new, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_valid_new, valid_prob_new), 4))

print("\nFeature Importance")
display(
    pd.DataFrame({
        "Feature": new_features,
        "Importance": enhanced_model.feature_importances_
    }).sort_values("Importance", ascending=False).round(4)
)


In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

gradient_model = HistGradientBoostingClassifier(
    max_iter=150,
    max_leaf_nodes=15,
    learning_rate=0.08,
    l2_regularization=1.0,
    random_state=42
)

gradient_model.fit(X_train_new, y_train_new)

gradient_pred = gradient_model.predict(X_valid_new)
gradient_prob = gradient_model.predict_proba(X_valid_new)[:, 1]

gradient_results = pd.DataFrame([{
    "Model": "HistGradientBoosting",
    "Accuracy": accuracy_score(y_valid_new, gradient_pred),
    "Precision": precision_score(
        y_valid_new, gradient_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_valid_new, gradient_pred, zero_division=0
    ),
    "F1": f1_score(y_valid_new, gradient_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_valid_new, gradient_prob)
}])

display(gradient_results.round(4))


In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    threshold_pred = (gradient_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Predicted Declining %": round(threshold_pred.mean() * 100, 2),
        "Precision": precision_score(
            y_valid_new, threshold_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_valid_new, threshold_pred, zero_division=0
        ),
        "F1": f1_score(
            y_valid_new, threshold_pred, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)
display(threshold_results.round(4))



In [ ]:

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "F1": f1_score(y_valid_new, valid_pred_new),
        "ROC-AUC": roc_auc_score(y_valid_new, valid_prob_new),
        "Average Precision": average_precision_score(
            y_valid_new, valid_prob_new
        )
    },
    {
        "Model": "HistGradientBoosting",
        "F1": f1_score(
            y_valid_new,
            (gradient_prob >= 0.30).astype(int)
        ),
        "ROC-AUC": roc_auc_score(y_valid_new, gradient_prob),
        "Average Precision": average_precision_score(
            y_valid_new, gradient_prob
        )
    }
])

display(comparison.round(4))


In [ ]:

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

# 1. Define the features used for training
final_features = feature_cols + [
    "impression_change",
    "log_prev_impressions",
    "log_prev2_impressions",
    "clicks_per_impression"
]

# 2. Prepare training data (February 2025 to May 2026 labels)
X_final_train = train_enhanced[final_features].fillna(0)
y_final_train = train_enhanced["is_declining"]

# 3. Prepare June 2026 data
june_enhanced = june_data.copy()

# Match June column names with training column names
june_enhanced["prev_impressions"] = june_enhanced["imp_may"]
june_enhanced["prev2_impressions"] = june_enhanced["imp_apr"]
june_enhanced["prev_clicks"] = june_enhanced["clicks_may"]
june_enhanced["prev_position"] = june_enhanced["pos_may"]

# Create the same additional features used during training
june_enhanced["impression_change"] = (
    june_enhanced["prev_impressions"]
    - june_enhanced["prev2_impressions"]
) / (june_enhanced["prev2_impressions"] + 1)

june_enhanced["log_prev_impressions"] = np.log1p(
    june_enhanced["prev_impressions"].clip(lower=0)
)

june_enhanced["log_prev2_impressions"] = np.log1p(
    june_enhanced["prev2_impressions"].clip(lower=0)
)

june_enhanced["clicks_per_impression"] = (
    june_enhanced["prev_clicks"]
    / (june_enhanced["prev_impressions"] + 1)
)

# 4. Select June features in the exact training order
X_june = june_enhanced[final_features].fillna(0)
y_june = june_enhanced["is_declining"]

# 5. Train the final Random Forest model
final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_final_train, y_final_train)

# 6. Make predictions for June
june_pred = final_model.predict(X_june)
june_prob = final_model.predict_proba(X_june)[:, 1]

# 7. Evaluate the model
print("Final Model: Random Forest")
print("June 2026 Test Results")
print("-" * 35)
print("Test rows:", f"{len(y_june):,}")
print("Accuracy:", round(accuracy_score(y_june, june_pred), 4))
print("Precision:", round(
    precision_score(y_june, june_pred, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_june, june_pred, zero_division=0), 4
))
print("F1 Score:", round(
    f1_score(y_june, june_pred, zero_division=0), 4
))
print("ROC-AUC:", round(roc_auc_score(y_june, june_prob), 4))
print("Average Precision:", round(
    average_precision_score(y_june, june_prob), 4
))

print("\nClassification Report:")
print(classification_report(y_june, june_pred, zero_division=0))


In [ ]:

# Create a content-priority table
priority_df = june_enhanced[
    ["client_hash_id", "content_hash_id", "is_declining"]
].copy()

priority_df["decline_probability"] = june_prob
priority_df["predicted_declining"] = june_pred

# Sort from highest to lowest predicted decline probability
priority_df = priority_df.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

# Show the top 20 content items
display(priority_df.head(20))

# Save the complete ranking
priority_df.to_csv(
    "june_2026_content_priority.csv",
    index=False
)

print("Saved: june_2026_content_priority.csv")
print("Total content rows:", len(priority_df))


In [ ]:

# Summarize the highest-priority content groups
priority_df["priority_group"] = pd.cut(
    priority_df["decline_probability"],
    bins=[-0.01, 0.30, 0.50, 0.70, 0.90, 1.00],
    labels=[
        "Very Low (below 0.30)",
        "Low (0.30-0.50)",
        "Medium (0.50-0.70)",
        "High (0.70-0.90)",
        "Very High (above 0.90)"
    ]
)

priority_summary = (
    priority_df.groupby("priority_group", observed=False)
    .agg(
        content_count=("content_hash_id", "count"),
        actual_decline_rate=("is_declining", "mean"),
        average_predicted_probability=("decline_probability", "mean")
    )
    .reset_index()
)

priority_summary["actual_decline_rate"] = (
    priority_summary["actual_decline_rate"] * 100
).round(2)

priority_summary["average_predicted_probability"] = (
    priority_summary["average_predicted_probability"] * 100
).round(2)

display(priority_summary)


In [ ]:

import os

for folder in ["work", "submission"]:
    print(f"\n--- {folder}/ ---")

    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            dirs[:] = [d for d in dirs if d != ".git"]
            for file in files:
                print(os.path.join(root, file))
    else:
        print("Folder not found")


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


In [ ]:

print("Available columns:")
print(data.columns.tolist())

print("\nFeature rows:", len(data))
print("Unique clients:", data["client_hash_id"].nunique())


In [ ]:

data = features.copy()

data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

data_query = data.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print("Rows:", len(data_query))
print("Unique clients:", data_query["client_hash_id"].nunique())
print("Available query features:")
print([
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
])
print(data_query[
    ["visible_queries", "rare_share", "anon_share", "top_query_share"]
].isna().mean().round(3))


In [ ]:
print("Feature rows:", len(data))
print("Unique clients:", data["client_hash_id"].nunique())
print("Available columns:", data.columns.tolist())
print("Missing values:")
print(data.isna().mean().round(3))

In [ ]:

data_query = features.copy()

if "is_declining" not in data_query.columns:
    data_query["is_declining"] = (
        data_query["imp_last30"]
        < 0.8 * data_query["imp_prev30"]
    ).astype(int)

data_query = data_query.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print("Label exists:", "is_declining" in data_query.columns)
print("Rows:", len(data_query))
print("Columns:", data_query.columns.tolist())


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

feature_cols = [
    "imp_prev30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data_query.dropna(
    subset=feature_cols + ["client_hash_id", "is_declining"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print("Overlapping clients:", len(
    set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
))

print("\nBaseline accuracy:",
      round(accuracy_score(y_test, baseline.predict(X_test)), 3))
print("\nModel accuracy:",
      round(accuracy_score(y_test, model.predict(X_test)), 3))
print("\nModel report:")
print(classification_report(
    y_test, model.predict(X_test), digits=3, zero_division=0
))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [ ]:

import os

repo = "/content/flyrank-ml-internship"

files_to_read = [
    "skills/README.md",
    "skills/writing-research-papers/SKILL.md",
    "DATA_USE.md",
    "work/README.md"
]

for relative_path in files_to_read:
    path = os.path.join(repo, relative_path)

    print("\n" + "=" * 65)
    print(relative_path)
    print("=" * 65)

    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("File not found")

print("\n" + "=" * 65)
print("SUBMISSION FILES")
print("=" * 65)

submission_path = os.path.join(repo, "submission")

for root, dirs, files in os.walk(submission_path):
    for filename in files:
        print(os.path.relpath(
            os.path.join(root, filename), repo
        ))


In [ ]:

import os
import json

repo = "/content/flyrank-ml-internship"

# Read the capstone report template and submission instructions
for relative_path in [
    "work/capstone_report_template.md",
    "submission/README.md",
    "submission/paper_url.txt"
]:
    path = os.path.join(repo, relative_path)

    print("\n" + "=" * 60)
    print(relative_path)
    print("=" * 60)

    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("File not found")

# Inspect the capstone notebook's markdown sections
notebook_path = os.path.join(
    repo, "work", "notebooks", "capstone.ipynb"
)

print("\n" + "=" * 60)
print("CAPSTONE NOTEBOOK")
print("=" * 60)

if os.path.isfile(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") == "markdown":
            source = "".join(cell.get("source", []))
            print(f"\n--- Markdown cell {i} ---")
            print(source[:1500])
else:
    print("Capstone notebook not found")


In [ ]:

import os

repo = "/content/flyrank-ml-internship"

paths = [
    "work/capstone_report_template.md",
    "work/notebooks/capstone.ipynb",
    "work/figures",
    "work/outputs",
    "submission/paper_url.txt",
]

for relative_path in paths:
    full_path = os.path.join(repo, relative_path)
    if os.path.isfile(full_path):
        status = "File exists"
    elif os.path.isdir(full_path):
        status = "Folder exists"
    else:
        status = "Missing"
    print(f"{relative_path}: {status}")


In [ ]:

import os
import json

repo = "/content/flyrank-ml-internship"

# Create the missing working folders
os.makedirs(os.path.join(repo, "work", "figures"), exist_ok=True)
os.makedirs(os.path.join(repo, "work", "outputs"), exist_ok=True)

report = r"""# Content Decline Triage: A Directional Machine Learning Study

- **Author:** Karim Yasser
- **Lane:** Supervised Machine Learning / Content Performance
- **Repo:** https://github.com/karim-yasser/flyrank-ml-internship
- **Date:** 2026-10-29

## 0. Abstract

This study asks whether recent content-performance history can help prioritize content items for review when impressions decline. It uses the gated FlyRank internship warehouse, including daily content-performance records from January 2025 through June 2026, and evaluates 92,375 eligible content items for June 2026. A Random Forest classifier uses lagged impressions, clicks, position, and derived historical features, with a temporal validation design and an all-declining baseline. On the June evaluation set, the model achieved 71.34% precision, 61.42% recall, and a ROC-AUC of 0.5635, while the baseline achieved 68.73% precision and 100% recall by flagging every item. These results suggest limited ranking discrimination, so the output should be treated as directional decision support for human review rather than a reliable prediction or proof that refreshing content will improve performance.

## 1. Problem framing

Content editors need a practical way to decide which pages to inspect first when performance changes. Reviewing every page equally may be inefficient, but an automated score can also create false alarms or overlook genuine declines.

The unit of analysis is a content item observed over monthly performance windows. The output is a decline probability and a ranked review queue. A human editor should inspect high-priority items alongside context such as content quality, recent changes, and business importance before deciding what to do.

A false positive may waste review time. A false negative may leave a declining item unreviewed. The model therefore supports prioritization, not automatic content changes.

## 2. Data safety

The analysis used the approved `FlyRank/internship-warehouse` release through the warehouse workflow. The daily performance table covered 2025-01-27 through 2026-06-30. The warehouse also included content and client dimension tables and a query-performance table.

The June evaluation frame contained 92,375 content-month rows across 40 clients after eligibility filters. The training frame contained 401,299 rows across 40 clients, covering target months from 2025-04-01 through 2026-05-01.

The query-performance table was not used to build the June target because its available window overlapped June and would not provide a clean, independent historical feature window for this evaluation. The model used historical performance values only. Pseudonymous client and content IDs were retained only for grouping, joining, and review-queue bookkeeping; they were not model features.

No client names, URLs, raw search queries, or private identifiers should appear in the public report. Do not commit the row-level June priority CSV or any warehouse extracts. Only aggregate results and safe summary artifacts belong in the public repository.

## 3. Baseline

The baseline flags every eligible item as declining. This simple rule is a useful reference because it captures the majority class and achieves 100% recall, but it cannot rank items by their individual risk.

On the June evaluation set, 68.73% of items met the decline definition. The baseline therefore achieved 68.73% accuracy, 68.73% precision, 100% recall, and an F1-score of 81.47%.

Because the baseline predicts the majority class for every item, its high recall and F1-score should not be mistaken for useful discrimination between individual items.

## 4. Model / analysis

The task is binary classification. An item is labelled as declining when June impressions are less than 80% of May impressions, among rows that satisfy the analysis eligibility rules.

The selected model is a Random Forest classifier with 100 trees, maximum depth 15, minimum leaf size 50, balanced class weights, random state 42, and parallel execution enabled.

The model uses these historical features:
- Previous-month impressions (`prev_impressions`)
- Impressions from two months earlier (`prev2_impressions`)
- Previous-month clicks (`prev_clicks`)
- Previous-month average position (`prev_position`)
- Relative impression change between the two historical months
- Log-transformed previous-month impressions
- Log-transformed impressions from two months earlier
- Previous-month clicks divided by previous-month impressions plus one

The target-month impressions are not used as features. Lag features are created only when the months are consecutive, to avoid comparing non-adjacent periods as if they were consecutive. Pseudonymous IDs are not model features.

The model was compared with a simple-feature Random Forest and an enhanced-feature Random Forest during temporal validation. The enhanced model showed only a small improvement in ROC-AUC over the simple-feature model on validation data, so any advantage should be interpreted cautiously.

## 5. Evaluation

The model-selection validation used earlier target months for training and March–May 2026 for validation. The final evaluation used June 2026, which was held outside the model-selection validation period. The same June frame was used for the model and the all-declining baseline.

| Metric | Random Forest | All-declining baseline |
|---|---:|---:|
| Accuracy | 56.53% | 68.73% |
| Precision | 71.34% | 68.73% |
| Recall | 61.42% | 100.00% |
| F1-score | 66.01% | 81.47% |
| ROC-AUC | 56.35% | Not applicable |

The decline base rate was 68.73%. The model's precision was higher than the baseline's, but its accuracy, recall, and F1-score were lower. Its ROC-AUC of 0.5635 indicates weak separation between declining and non-declining items.

The model therefore does not demonstrate an overall performance improvement over the simple baseline. Its possible practical value is limited to offering a different review order, which would need to be assessed with a clear operational objective and further validation.

## 6. Interpretation

The results show that the available historical features contain some directional signal, but the signal is weak. A high predicted probability should not be interpreted as certainty that an item will decline, and a low score should not be used to rule out a review.

The priority groups showed a general increase in observed decline rate as predicted-risk groups increased. However, the low- and medium-risk groups had similar observed decline rates, and the very-high-risk group contained only 51 items. That small group is not enough to establish a stable pattern.

The results do not show that refreshing a page causes impressions to recover. No causal experiment was performed, and the study does not predict Google's ranking algorithm.

## 7. Ranked recommendations

1. **Use the queue only as a review aid.** Editors can consider higher-scored items first, but should confirm the context before taking action.
2. **Keep a baseline-based comparison.** The current model did not beat the all-declining baseline on accuracy or F1-score, so its usefulness should be measured against the actual review workflow.
3. **Review the cost of missed declines.** The model missed some declining items. If missing a decline is costly, the decision threshold and operational target need further evaluation.
4. **Collect outcome evidence.** Track what editors decide to change and measure subsequent performance. A causal claim about content refresh requires an appropriate experimental or causal design.
5. **Revalidate over time.** Repeat the evaluation on later months and monitor whether the ranking signal remains useful.

Confidence is low for individual predictions because ROC-AUC is close to 0.5. The results are directional and suitable for cautious decision support only.

## 8. Reproducibility

Repository: https://github.com/karim-yasser/flyrank-ml-internship

Notebook: `work/notebooks/capstone.ipynb`

The analysis was developed in Google Colab using Python, DuckDB, pandas, and scikit-learn. The selected Random Forest used `random_state=42`.

To reproduce the work, clone the repository, open the capstone notebook in Colab, install the required packages, and authenticate to the approved gated warehouse release using a private token stored in Colab Secrets or entered securely at runtime. Do not place tokens or dataset extracts in the repository.

The notebook must be run from top to bottom to verify the reported numbers. The exact warehouse-access steps, filters, feature construction, and evaluation code should remain in the committed notebook so that the results can be checked from a fresh run.

## 9. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

"""

report_path = os.path.join(repo, "work", "capstone_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

metrics = {
    "evaluation_period": "June 2026",
    "evaluation_rows": 92375,
    "decline_base_rate": 0.687286,
    "model": {
        "name": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_leaf": 50,
        "class_weight": "balanced",
        "random_state": 42,
        "accuracy": 0.5653,
        "precision": 0.7134,
        "recall": 0.6142,
        "f1": 0.6601,
        "roc_auc": 0.5635,
        "average_precision": 0.7365
    },
    "baseline": {
        "name": "Predict every item as declining",
        "accuracy": 0.6873,
        "precision": 0.6873,
        "recall": 1.0,
        "f1": 0.8147,
        "roc_auc": None
    },
    "notes": [
        "Directional decision support only.",
        "The model did not outperform the baseline on accuracy or F1.",
        "Re-run the notebook and verify these values before publication."
    ]
}

metrics_path = os.path.join(repo, "work", "outputs", "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Created:")
print(report_path)
print(metrics_path)
print("Figures folder:", os.path.join(repo, "work", "figures"))


In [ ]:

import os
import json

repo = "/content/flyrank-ml-internship"

# Check the report
report_path = os.path.join(repo, "work", "capstone_report.md")
with open(report_path, "r", encoding="utf-8") as f:
    report = f.read()

print("Report length:", len(report), "characters")
print("Report sections:")
for heading in [
    "## 0. Abstract",
    "## 1. Problem framing",
    "## 2. Data safety",
    "## 3. Baseline",
    "## 4. Model / analysis",
    "## 5. Evaluation",
    "## 6. Interpretation",
    "## 7. Ranked recommendations",
    "## 8. Reproducibility",
    "## 9. Acknowledgments & data credit"
]:
    print(("OK  " if heading in report else "MISSING  ") + heading)

# Check metrics file
metrics_path = os.path.join(repo, "work", "outputs", "metrics.json")
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("\nJune evaluation rows:", metrics["evaluation_rows"])
print("Model ROC-AUC:", metrics["model"]["roc_auc"])
print("Baseline F1:", metrics["baseline"]["f1"])

# Check figures folder
figures_path = os.path.join(repo, "work", "figures")
print("\nFigures folder exists:", os.path.isdir(figures_path))
print("Figures currently:", os.listdir(figures_path))


In [ ]:

import os
import matplotlib.pyplot as plt

repo = "/content/flyrank-ml-internship"
figures_dir = os.path.join(repo, "work", "figures")
os.makedirs(figures_dir, exist_ok=True)

# 1. Model vs baseline
metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
model_scores = [56.53, 71.34, 61.42, 66.01]
baseline_scores = [68.73, 68.73, 100.00, 81.47]

x = list(range(len(metrics)))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([i - width / 2 for i in x], model_scores, width, label="Random Forest")
ax.bar([i + width / 2 for i in x], baseline_scores, width, label="All-declining baseline")
ax.set_ylabel("Score (%)")
ax.set_title("June 2026: Model vs Baseline")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "model_vs_baseline.png"), dpi=180)
plt.show()
plt.close(fig)

# 2. June decline base rate
labels = ["Declining", "Not declining"]
values = [68.7286, 100 - 68.7286]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values)
ax.set_ylabel("Share of evaluated items (%)")
ax.set_title("June 2026 Evaluation Base Rate")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center"
    )

fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "june_base_rate.png"), dpi=180)
plt.show()
plt.close(fig)

# 3. Risk groups: predicted probability vs observed decline rate
groups = ["Very Low", "Low", "Medium", "High", "Very High"]
observed_decline = [45.04, 66.41, 69.58, 77.93, 96.08]
mean_probability = [26.26, 42.69, 59.24, 75.86, 90.92]
group_counts = [2591, 35127, 43212, 11394, 51]

fig, ax = plt.subplots(figsize=(10, 5))
x = list(range(len(groups)))
ax.plot(x, observed_decline, marker="o", label="Observed decline rate")
ax.plot(x, mean_probability, marker="s", label="Mean predicted probability")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("Percentage (%)")
ax.set_title("Predicted Risk Groups vs Observed Decline")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.25)

for i, count in enumerate(group_counts):
    ax.annotate(
        f"n={count:,}",
        (i, min(observed_decline[i], mean_probability[i]) - 7),
        ha="center",
        fontsize=8
    )

fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "risk_groups.png"), dpi=180)
plt.show()
plt.close(fig)

# Add the figures to the report in relevant sections
report_path = os.path.join(repo, "work", "capstone_report.md")

with open(report_path, "r", encoding="utf-8") as f:
    report = f.read()

insertions = [
    (
        "## 2. Data safety",
        "\n\n![June 2026 evaluation base rate](figures/june_base_rate.png)\n\n"
        "*Figure 1. The June evaluation set had a 68.73% observed decline rate.*\n"
    ),
    (
        "## 5. Evaluation",
        "\n\n![June 2026 model versus baseline](figures/model_vs_baseline.png)\n\n"
        "*Figure 2. The model had higher precision, but lower accuracy, recall, and F1-score than the all-declining baseline.*\n"
    ),
    (
        "## 6. Interpretation",
        "\n\n![Predicted risk groups and observed decline rates](figures/risk_groups.png)\n\n"
        "*Figure 3. Observed decline rates generally rise across risk groups, but the very-high-risk group is small (51 items).*\n"
    )
]

for heading, insertion in insertions:
    if insertion.strip() not in report:
        report = report.replace(heading, heading + insertion, 1)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("\nCreated figure files:")
for filename in sorted(os.listdir(figures_dir)):
    print(filename)

print("\nUpdated report:", report_path)


In [ ]:

import os
import json
import subprocess

repo = "/content/flyrank-ml-internship"

notebooks = [
    "work/notebooks/capstone.ipynb",
    "notebooks/03_working_with_the_full_release.ipynb"
]

for relative_path in notebooks:
    path = os.path.join(repo, relative_path)
    print("\n" + "=" * 65)
    print(relative_path)
    print("=" * 65)

    if not os.path.isfile(path):
        print("Notebook not found")
        continue

    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])
    print("Total cells:", len(cells))
    print("Code cells:", sum(c.get("cell_type") == "code" for c in cells))

    for i, cell in enumerate(cells):
        source = "".join(cell.get("source", []))
        if cell.get("cell_type") == "markdown":
            print(f"\nMarkdown cell {i}:")
            print(source[:250].replace("\n", " "))
        elif source.strip():
            print(f"Code cell {i}: {source[:120].replace(chr(10), ' ')}")

print("\n" + "=" * 65)
print("GIT STATUS")
print("=" * 65)

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo,
    capture_output=True,
    text=True
)
print(result.stdout if result.stdout.strip() else "No tracked changes detected")


In [ ]:

import os
import json

repo = "/content/flyrank-ml-internship"
path = os.path.join(repo, "work", "notebooks", "capstone.ipynb")

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    print(f"\n{'=' * 55}\nCELL {i} — {cell['cell_type']}\n{'=' * 55}")
    print(source[:2500])

    if cell["cell_type"] == "code":
        outputs = cell.get("outputs", [])
        print("Stored outputs:", len(outputs))
        for output in outputs:
            if "ename" in output:
                print("Previous error:", output["ename"], output.get("evalue", ""))

print("\nNotebook inspection complete.")


In [ ]:

import os
import json

repo = "/content/flyrank-ml-internship"
notebook_path = os.path.join(repo, "work", "notebooks", "capstone.ipynb")
metrics_path = os.path.join(repo, "work", "outputs", "metrics.json")

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

# Replace the skeleton's markdown sections with the full content.
markdown_content = {
    1: """## 1. Question

Can recent content-performance history help prioritize pages for human review when impressions decline?

The unit of analysis is a content item observed across monthly windows. The output is a decline-risk score and a ranked review queue. Editors should review the context before deciding whether to update a page. A false positive costs review time; a false negative may leave a declining page unchecked. This is decision support, not an automated publishing decision.""",

    3: """## 2. Data

The analysis used the approved `FlyRank/internship-warehouse` release through the DuckDB warehouse workflow.

- Daily performance covered 2025-01-27 through 2026-06-30.
- The training frame contained 401,299 rows across 40 clients, with target months from April 2025 through May 2026.
- The June 2026 evaluation frame contained 92,375 eligible rows across 40 clients.
- The query-level table was excluded from the June target model because its time window overlapped June and did not provide an independent historical feature window for this evaluation.

Pseudonymous IDs were used only for grouping, joins, and queue bookkeeping, never as model features. The public repository must not contain row-level warehouse exports, private client information, URLs, or raw queries.""",

    5: """## 3. Methodology

The target is positive when June impressions are less than 80% of May impressions, subject to the eligibility filters used in the analysis.

The selected model is a Random Forest classifier with 100 trees, maximum depth 15, minimum leaf size 50, balanced class weights, and random state 42.

Features:
- Previous-month impressions
- Impressions from two months earlier
- Previous-month clicks
- Previous-month average position
- Relative change between the two historical impression windows
- Log-transformed historical impressions
- Previous-month clicks per impression

Target-month impressions and pseudonymous identifiers were excluded from the feature set. Lag comparisons were restricted to consecutive months. Model selection used a temporal validation period from March through May 2026; June was used as the final evaluation period.

The baseline flags every item as declining. This baseline is simple but important because it captures the majority class.""",

    7: """## 4. Results (vs baseline)

The June decline base rate was 68.73%. The Random Forest had higher precision than the all-declining baseline, but lower accuracy, recall, and F1-score. Its ROC-AUC was 0.5635, indicating weak separation between declining and non-declining items.

The model did not outperform the baseline on accuracy or F1-score. Any potential value is limited to helping prioritize a human review queue and needs further operational validation.""",

    9: """## 5. Limitations

- The ROC-AUC is close to 0.5, so individual scores have limited discriminatory value.
- The model missed some declining items; recall was 61.42%, compared with 100% for the all-declining baseline.
- The baseline's high F1-score reflects the high prevalence of declining items and should not be interpreted as useful ranking.
- The very-high-risk group contained only 51 items, so its observed rate is not enough to establish a stable pattern.
- The study does not establish that refreshing content causes impressions to recover.
- It does not predict Google's ranking algorithm.
- Results may change on future months, other clients, or different eligibility rules.""",

    11: """## 6. Ranked recommendations

1. Use the score only to help order human reviews; do not automate content changes.
2. Have editors verify context, content quality, recent changes, and business importance before acting.
3. Compare the queue against the all-declining baseline using an operational measure such as useful reviews per hour.
4. Review false negatives if missed declines are costly, and assess alternative thresholds on validation data before changing the operating point.
5. Re-evaluate on later months and track outcomes. Claims that refreshes cause improvement require an appropriate experiment or causal design.

Confidence in individual predictions is low. These recommendations are directional decision support only.""",

    13: """## 7. Artifacts the paper embeds

The figures below summarize the June base rate, model-versus-baseline metrics, and observed decline rates across predicted-risk groups. They use aggregate results rather than row-level content records.

The metrics file is `work/outputs/metrics.json`; the figure files are stored under `work/figures/`. Verify all values against a fresh run of the analysis notebook before publication.""",

    15: """## ML-12: 5-minute demo outline

- **0:00–0:45 — Problem:** explain why editors need a careful way to prioritize content reviews.
- **0:45–1:30 — Data and safety:** summarize the approved warehouse, time windows, and the exclusion of identifiers from model features.
- **1:30–2:30 — Method:** explain the 20% month-over-month decline label, historical features, temporal validation, and all-declining baseline.
- **2:30–3:30 — Results:** show the comparison chart and explain that precision improved slightly, while accuracy and F1 did not beat the baseline.
- **3:30–4:15 — Limitations:** emphasize weak ROC-AUC and the lack of causal evidence.
- **4:15–5:00 — Recommendation:** demonstrate how a human might review a prioritized queue, then describe the next validation steps.

### Social-post draft

I built a content-decline triage prototype using historical performance data and a time-aware evaluation. The Random Forest achieved 71.34% precision on the June 2026 evaluation set, but its ROC-AUC was 0.5635 and it did not beat an all-declining baseline on accuracy or F1-score. The takeaway: scores may support cautious human review, but they are not proof of future outcomes or content-refresh impact. [Project repository](https://github.com/karim-yasser/flyrank-ml-internship)

### Employer-facing summary

I built a supervised machine-learning prototype to help prioritize content items for performance review using historical impressions, clicks, and position features. I evaluated a Random Forest against an all-positive baseline with a temporal validation design and a June 2026 evaluation set. The results showed limited discrimination, so I framed the output as cautious decision support and documented its limitations rather than overstating model performance.

### Self-check

- [ ] Full analysis notebook runs top to bottom without errors.
- [ ] Capstone figures and metrics match a fresh analysis run.
- [ ] No row-level data, client identifiers, URLs, or raw queries are committed.
- [ ] Paper includes abstract, all required sections, and data credit.
- [ ] Repository changes are committed and pushed.
- [ ] Paper is deployed and its exact URL is saved in `submission/paper_url.txt`.
- [ ] Demo outline, social post, and employer-facing summary are complete."""
}

for index, content in markdown_content.items():
    nb["cells"][index]["source"] = content.splitlines(keepends=True)

# Fill the existing code cells with checks and visual outputs.
code_content = {
    2: """from IPython.display import display, Markdown
import os
import json

REPO = "/content/flyrank-ml-internship"
METRICS_PATH = os.path.join(REPO, "work", "outputs", "metrics.json")
FIGURES_DIR = os.path.join(REPO, "work", "figures")

with open(METRICS_PATH, "r", encoding="utf-8") as f:
    metrics = json.load(f)

display(Markdown("**Loaded aggregate evaluation metrics.**"))
print("Evaluation period:", metrics["evaluation_period"])
print("Evaluation rows:", f'{metrics["evaluation_rows"]:,}')
print("Decline base rate:", f'{metrics["decline_base_rate"]:.2%}')
""",

    4: """print("Data window: 2025-01-27 to 2026-06-30")
print("Training rows: 401,299")
print("Training clients: 40")
print("Training target months: 2025-04-01 to 2026-05-01")
print("June evaluation rows:", f'{metrics["evaluation_rows"]:,}')
print("June evaluation clients: 40")
print("Pseudonymous IDs are not model features.")
print("Do not commit row-level exports or the June priority CSV.")
""",

    6: """model = metrics["model"]
baseline = metrics["baseline"]

print("Model:", model["name"])
print("Parameters: n_estimators=100, max_depth=15, min_samples_leaf=50")
print("class_weight=balanced, random_state=42")
print("Baseline:", baseline["name"])
print("Target: June impressions < 80% of May impressions.")
print("Validation: temporal validation on March–May 2026; final evaluation on June 2026.")
""",

    8: """from IPython.display import display, Markdown
import pandas as pd

model = metrics["model"]
baseline = metrics["baseline"]

results = pd.DataFrame([
    {
        "Metric": "Accuracy",
        "Random Forest": model["accuracy"],
        "All-declining baseline": baseline["accuracy"]
    },
    {
        "Metric": "Precision",
        "Random Forest": model["precision"],
        "All-declining baseline": baseline["precision"]
    },
    {
        "Metric": "Recall",
        "Random Forest": model["recall"],
        "All-declining baseline": baseline["recall"]
    },
    {
        "Metric": "F1-score",
        "Random Forest": model["f1"],
        "All-declining baseline": baseline["f1"]
    },
    {
        "Metric": "ROC-AUC",
        "Random Forest": model["roc_auc"],
        "All-declining baseline": None
    }
])

display(results.style.format({
    "Random Forest": "{:.2%}",
    "All-declining baseline": lambda x: "N/A" if pd.isna(x) else f"{x:.2%}"
}))

print("Average precision:", f'{model["average_precision"]:.2%}')
print("Base rate:", f'{metrics["decline_base_rate"]:.2%}')
""",

    10: """print("Main limitation: ROC-AUC is close to 0.5.")
print("The model does not beat the baseline on accuracy or F1-score.")
print("The evaluation does not establish causal effects of content refresh.")
print("These are aggregate reported metrics; rerun the analysis notebook to verify them.")
""",

    12: """print("Recommended use: cautious human review prioritization.")
print("Do not automatically update or remove content based on the score.")
print("Re-evaluate performance on later months before operational use.")
""",

    14: """from IPython.display import display, Image, Markdown

figure_files = [
    ("june_base_rate.png", "June 2026 evaluation base rate"),
    ("model_vs_baseline.png", "Model versus baseline"),
    ("risk_groups.png", "Predicted risk groups and observed decline rates")
]

for filename, caption in figure_files:
    path = os.path.join(FIGURES_DIR, filename)
    if os.path.isfile(path):
        display(Markdown(f"**{caption}**"))
        display(Image(filename=path))
    else:
        print("Missing figure:", filename)

print("Metrics file:", METRICS_PATH)
print("Figures folder:", FIGURES_DIR)
"""
}

for index, content in code_content.items():
    nb["cells"][index]["source"] = content.splitlines(keepends=True)
    nb["cells"][index]["outputs"] = []
    nb["cells"][index]["execution_count"] = None

# Add ML-12 closing cells before the self-check.
if not any(
    "ML-12: 5-minute demo outline" in "".join(c.get("source", []))
    for c in nb["cells"]
):
    self_check_index = next(
        i for i, c in enumerate(nb["cells"])
        if "## Self-check" in "".join(c.get("source", []))
    )
    nb["cells"].insert(self_check_index, {
        "cell_type": "markdown",
        "metadata": {},
        "source": markdown_content[15].splitlines(keepends=True)
    })

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print("Updated capstone notebook:")
print(notebook_path)
print("Markdown/code sections filled:", len(markdown_content) + len(code_content))
print("Important: this notebook summarizes saved metrics; it does not retrain the model.")


In [54]:
import os

repo = "/content/flyrank-ml-internship"

paths = [
    "work/capstone_report.md",
    "work/notebooks/capstone.ipynb",
    "work/outputs/metrics.json",
    "work/figures/june_base_rate.png",
    "work/figures/model_vs_baseline.png",
    "work/figures/risk_groups.png",
    "submission/paper_url.txt",
]

for path in paths:
    full_path = os.path.join(repo, path)
    print(("✅" if os.path.exists(full_path) else "❌"), path)

✅ work/capstone_report.md
✅ work/notebooks/capstone.ipynb
✅ work/outputs/metrics.json
✅ work/figures/june_base_rate.png
✅ work/figures/model_vs_baseline.png
✅ work/figures/risk_groups.png
✅ submission/paper_url.txt
